In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import numpy as np

In [ ]:
N = 1000
P = 100

stats_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats/')

results_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/results/concat/')
dataset_folder = f"syn_data_N{N}_P{P}"
dm_file = 'datamatrix__3configs__st_0.75q_dt_0.25q.parquet'

data_path = os.path.join(results_path,dataset_folder,dm_file)

In [ ]:
data_matrix = pd.read_parquet(data_path)

In [ ]:
# rem duplicate cols
dup = data_matrix.T.duplicated()
df_dedup = data_matrix.loc[:, ~dup]

# rem constant cols
df_dedup = df_dedup.loc[:,~(df_dedup.nunique() <= 1)]

df_dedup_scaled = (df_dedup - df_dedup.mean())/df_dedup.std()
print(f"dropped {dup.sum()} duplicate columns")

In [ ]:
df_dedup_names = df_dedup.columns.str.replace(stats_path, ".", regex=False)
df_dedup.columns = df_dedup_names

In [ ]:
data_matrix.shape, df_dedup.shape

In [ ]:
pyspoc_stat_fam = pd.Series(data_matrix.columns.str.split(".").str[-6:-1].str.join("."), index=data_matrix.columns)

In [ ]:
# pyspoc_stat_fam.reset_index().groupby(0)['index'].apply(list)

In [ ]:
# the data matrix is quite bloated with 'Moment' features right now
# i think that reducer needs to be changed. im just dropping it for now.

is_moment_vec = data_matrix.columns.str.contains(r"Moment\.\w+_\d+$", regex=True)
df_no_moment = data_matrix.loc[:, ~is_moment_vec]
print(f"dropped {is_moment_vec.sum()} per-column moment features")

In [ ]:
# df_moment

In [ ]:
df_moment = data_matrix.loc[:,is_moment_vec]

In [ ]:
# pyspoc_stat_fam

In [ ]:
pyspoc_stat_fam[df_moment.columns].reset_index().groupby(0)['index'].count()
# apply(list).iloc[0]

In [ ]:
df_moment

In [ ]:
# oas_m = df_dedup.filter(like="EllipticEnvelope.ee__pyspoc.reducers.basic.Moment")
# r = oas_m.corr(method="spearman").abs()

In [ ]:
# code to look at spearman correlations
# corr = df_dedup.corr(method="spearman")
# upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
# pairs = upper.stack().sort_values(key=abs, ascending=False)
# exact = pairs[pairs.abs() == 1.0]
# print(len(exact), "pairs with corr = 1")
# print(exact.head(30))

In [ ]:
df_module_cluster = df_no_moment

In [ ]:
plt.figure(figsize=(12, 8))
sns.clustermap(df_module_cluster.fillna(0), cmap="RdBu_r", center=0, vmin=-3, vmax=3,
               xticklabels=False, yticklabels=False, figsize=(12, 10))

plt.xlabel("statistics"); plt.ylabel("datasets")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_module_cluster.fillna(0), cmap="RdBu_r", center=0, vmin=-3, vmax=3,
               xticklabels=False, yticklabels=False)

plt.xlabel("statistics"); plt.ylabel("datasets")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [ ]:
corr = df_module_cluster.corr(method='spearman')
abs_corr = corr.abs()

In [ ]:
dissim_link= linkage(squareform(1-abs_corr,checks=False),method='average')

In [ ]:
# find clusters/modules in the data
modules = pd.Series(fcluster(dissim_link,t=0.3,criterion='distance'), index=abs_corr.index)

In [ ]:
# modules.value_counts().value_counts()

In [ ]:
print("found " , modules.nunique(), "modules out of ", df_module_cluster.shape[1], " features")
print(f'module id with size..\n{modules.value_counts()[:3]}')
print(f'module size counts\n{modules.value_counts().value_counts()}')

In [ ]:
for m in modules.value_counts().index[:5]:
    members = modules.index[modules == m]
    print(f"module {m} ({len(members)} features):")
    print(pyspoc_stat_fam[members].value_counts(), "\n")

In [ ]:
# sp.moment(cols,axis=1,order=[1,2,3]).mean(axis=1)
# n = data.shape[0]
# cols = data[~np.eye(n, dtype=bool)].reshape(n, n - 1)     # row i = column i of data, without its diagonal entry
# import numpy as np
# import scipy.stats as sp

# rng = np.random.default_rng(0)
# data = rng.standard_normal((100, 100))   # stand-in for a 5 x 4 matrix

# print(sp.moment(data, order=[1]))  # default axis=0: one value per column
# print(sp.moment(data, order=[1], axis=None))  # whole matrix: one value
# print(sp.moment(data, order=[1, 2, 3]).mean(axis=1))